In [ ]:
import sys
if 'google.colab' in sys.modules:
    print('Running on Google Colab')

In [ ]:
# Cell 1 — install missing dependencies (keras and tensorflow are already on Colab)
!pip install -q rasterio wandb

In [ ]:
# Cell 2 — download and unzip data
wget -q "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"
!unzip -q EuroSAT_RGB.zip -d data/

In [ ]:
# Cell 3 — run training
import sys
sys.path.insert(0, '..')

from src.dataset import load_rgb_dataset
from src.model import build_model
import keras, wandb
from wandb.integration.keras import WandbMetricsLogger

train_ds, val_ds = load_rgb_dataset("data/EuroSAT_RGB")
model = build_model(in_channels=3)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
wandb.init(project="eurosat-cnn", name="RGB_baseline")
model.fit(train_ds, epochs=50, validation_data=val_ds,
          callbacks=[WandbMetricsLogger(),
                     keras.callbacks.ModelCheckpoint("RGB_baseline_best.keras",
                                                     monitor="val_accuracy",
                                                     save_best_only=True)])
wandb.finish()